# Tests de l'API FastAPI

Ce notebook vérifie les deux endpoints attendus de l'API :

- `POST /predict`
- `POST /recommend`

Prérequis : lancer l'API avant d'exécuter ce notebook.

```bash
uvicorn main:app --reload
```


In [34]:
import json
from pathlib import Path
from urllib import error, request
import pandas as pd

dataset = pd.read_csv("data/dataset_consolide.csv")
sample_row = dataset.dropna(subset=["average_rain_fall_mm_per_year", "avg_temp"]).iloc[0]

## Payloads

  ### POST /predict
  Prend en entrée les données d'une seule culture et son contexte, et retourne la prédiction de rendement.

In [ ]:
predict_payload = {
    "area": "Afghanistan",
    "crop": "Maize",
    "year": 1995, # dans streamlit, on forcera l'année en cours
    "average_rain_fall_mm_per_year": 327.0,
    "pesticides_tonnes": None,
    "avg_temp": 15.0,
}

### POST /recommend
Prend en entrée uniquement le contexte (température, etc.), et retourne une liste de toutes les cultures possibles, triées par rendement prédit décroissant.

In [ ]:
recommend_payload = {
    "area": "Afghanistan",
    "year": 1995, # dans streamlit, on forcera l'année en cours
    "average_rain_fall_mm_per_year": 327.0,
    "pesticides_tonnes": None,
    "avg_temp": 15.0,
    # Optionnel : restreindre la recommandation à certaines cultures
    # "candidate_crops": ["Maize", "Wheat", "Rice, paddy"],
}

## Fonction utilitaire HTTP

Cette fonction envoie une requête JSON à l'API et renvoie le code HTTP et le corps de réponse.


In [36]:
def post_json(endpoint, payload):
    body = json.dumps(payload).encode("utf-8")
    req = request.Request(
        f"http://127.0.0.1:8000{endpoint}",
        data=body,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    try:
        with request.urlopen(req) as response:
            response_body = response.read().decode("utf-8")
            return response.status, json.loads(response_body)
    except error.URLError as exc:
        raise RuntimeError(
            "Impossible de joindre l'API. Lancer `uvicorn main:app --reload` avant d'exécuter ce notebook."
        ) from exc


## Test de `POST /predict`


In [37]:
predict_status, predict_response = post_json("/predict", predict_payload)

assert predict_status == 200
assert "predicted_yield_t_ha" in predict_response

{
    "payload": predict_payload,
    "response": predict_response,
}


{'payload': {'area': 'Afghanistan',
  'crop': 'Maize',
  'year': 1995,
  'average_rain_fall_mm_per_year': 327.0,
  'pesticides_tonnes': None,
  'avg_temp': 15.0},
 'response': {'predicted_yield_t_ha': 1.6106}}

## Test de `POST /recommend`


In [38]:
recommend_status, recommend_response = post_json("/recommend", recommend_payload)
recommendations = recommend_response["recommendations"]

assert recommend_status == 200
assert len(recommendations) > 0
assert recommendations == sorted(
    recommendations,
    key=lambda row: row["predicted_yield_t_ha"],
    reverse=True,
)

pd.DataFrame(recommendations).head(10)


,crop,predicted_yield_t_ha
0,Potatoes,16.6324
1,Sweet potatoes,11.9558
2,Cassava,8.6598
3,Yams,7.0485
4,Plantains and others,6.3612
5,"Rice, paddy",2.0755
6,Maize,1.6106
7,Soybeans,1.4491
8,Wheat,1.1049
9,Sorghum,0.7049
